In [ ]:
#IMPORTANDO PORTFOLIOENV DO TREINAMENTO_RL

import gymnasium as gym
import numpy as np
from gymnasium import spaces    

class PortfolioEnv(gym.Env):
    metadata = {'render.modes': ['human']}

    def __init__(self, features, returns, lambda_risk=0.5, turnover_cost=0.001):
        super(PortfolioEnv, self).__init__()

        self.features = features.values # Matriz de estados
        self.returns = returns.values # Matriz de retornos reais dos ativos
        self.asset_names = returns.columns

        self.n_assets = returns.shape[1]
        self.n_features = features.shape[1]
        self.current_step = 0

        # Hiperparâmetros da Recompensa 
        self.lambda_risk = lambda_risk      # Penalidade por volatilidade
        self.turnover_cost = turnover_cost  # Custo de transação/mudança

        # Espaço de Ação: vetor contínuo de pesos (0 a 1) para cada ativo
        self.action_space = spaces.Box(low=0, high=1, shape=(self.n_assets,), dtype=np.float32)
        
        # Espaço de Observação
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.n_features,), dtype=np.float32)
        
        # Estado inicial dos pesos (começa 100% em caixa ou distribuído igualmente)
        self.last_weights = np.ones(self.n_assets) / self.n_assets

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.last_weights = np.ones(self.n_assets) / self.n_assets
        
        # Retorna a primeira observação e info vazia
        return self.features[self.current_step], {}

    def step(self, action):
        # Normalizar ação para garantir soma = 1 (Alocação de Portfólio) 
        weights = np.exp(action) / np.sum(np.exp(action))
        
        # Pegar o retorno real do dia atual (o que aconteceu no mercado)
        daily_returns = self.returns[self.current_step]
        
        # Calcular retorno do portfólio: soma(pesos * retorno do ativo)
        portfolio_return = np.sum(weights * daily_returns)
        
        # Calcular custo de turnover (penalidade por mudar muito os pesos)
        turnover = np.sum(np.abs(weights - self.last_weights))
        
        # Calcular risco (volatilidade instantânea aproximada pelo retorno quadrático ou janela)
        # Simplificação para recompensa instantânea: Risco ~ (Retorno do portfólio)^2 ou volatilidade recente
        portfolio_risk = portfolio_return ** 2 # Proxy simples de variância local
        
        # Função de recompensa 
        # Reward = Retorno - (lambda * Risco) - (custo * turnover)
        reward = portfolio_return - (self.lambda_risk * portfolio_risk) - (self.turnover_cost * turnover)
        
        # Atualizar estado
        self.last_weights = weights
        self.current_step += 1
        
        # Verificar se acabou os dados
        terminated = self.current_step >= len(self.features) - 1
        truncated = False
        
        # Próximo estado
        next_observation = self.features[self.current_step]
        
        info = {
            'portfolio_return': portfolio_return,
            'turnover': turnover,
            'weights': weights
        }
        
        return next_observation, reward, terminated, truncated, info

    def render(self, mode='human'):
        pass

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from joblib import Parallel, delayed
import torch

print(f"Acelerando com todos os núcleos de processamento...")


df_features = pd.read_csv("features_rl_sp100.csv", index_col=0, parse_dates=True)

#Puxa os retornos baseado na variação, não preço absoluto
df_returns = pd.read_csv("matriz_final_v3_survivorship.csv", index_col=0, parse_dates=True).pct_change().dropna()

#Mantém as datas em comum
df_features, df_returns = df_features.align(df_returns, join='inner', axis=0)


# WALK FORWARD BACKTEST -----------------------------------------------
#Vamos fazer um backtest simulando a vida real, essa simulação consiste em treinar o agente em dados de 2 anos do início da nossa base, 
#testar (operar) por 3 meses e re-treinar ele baseado no resultado da operação.  
# Treino: 2 anos =~ 504 dias úteis)
def executar_walk_forward(features, returns, verbose=False): 
    janela_treino = 504 # ~2 anos em dias uteis
    janela_teste = 63 # ~3 meses em dias uteis
    dias_totais = len(features)
    start_index = janela_treino
    
    wf_resultado = []
    
    # Loop Walk-Forward 
    # Range da data inicial até o final, indo de 3 em 3 meses.
    for t in range(start_index, dias_totais, janela_teste):
        train_start = t - janela_treino     
        train_end = t
        test_end = min(t + janela_teste, dias_totais)
        
        if test_end <= train_end: break
        
        if verbose:
            print(f"Ciclo: {features.index[train_start].date()} -> {features.index[train_end-1].date()}")
        
        # Recortes
        X_train = features.iloc[train_start:train_end]
        y_train = returns.iloc[train_start:train_end]
        X_test = features.iloc[train_end:test_end]
        y_test = returns.iloc[train_end:test_end]
        
        # Treino
        env_train = DummyVecEnv([lambda: PortfolioEnv(X_train, y_train, lambda_risk=0.5)])
        model = PPO("MlpPolicy", env_train, verbose=0, seed=42)
        model.learn(total_timesteps=10000) 
        
        # Teste
        env_test = PortfolioEnv(X_test, y_test)
        obs, _ = env_test.reset()
        done = False
        
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, terminated, truncated, _ = env_test.step(action)
            
            # Retorno Real
            idx = env_test.current_step - 1
            r_dia = np.dot(action, y_test.iloc[idx].values)
            wf_resultado.append(r_dia)
            done = terminated or truncated

    # Cálculo do Sharpe Final da Estratégia
    rets = np.array(wf_resultado)
    if len(rets) == 0 or np.std(rets) == 0: return -999.0
    
    sharpe_final = (np.mean(rets) / np.std(rets)) * np.sqrt(252)
    return sharpe_final

In [ ]:
# USAREMOS PARALELISMO PRA ACELERAR O CÁLCULO DA PERMUTAÇÃO
#Vamos realizar o Walk Forward e posteriormente realizar a permutação para testar a robustez do modelo
print("~ Calculando Performance Real ~")
sharpe_real = executar_walk_forward(df_features, df_returns, verbose=True)
print(f"Sharpe Ratio Walk-Forward: {sharpe_real:.4f}")

#WALK FORWARD PERFORMANCE ---------------------------------------------------------------
df_wf = pd.DataFrame(wf_resultado).set_index('Date')
df_wf['Cumulative_Return'] = (1 + df_wf['Return']).cumprod()

# Benchmark (Média do mercado no mesmo período)
market_slice = df_returns.loc[df_wf.index]
market_return = (1 + market_slice.mean(axis=1)).cumprod()

# Métricas Finais
total_ret_algo = df_wf['Cumulative_Return'].iloc[-1] - 1
total_ret_market = market_return.iloc[-1] - 1
sharpe_algo = (df_wf['Return'].mean() / df_wf['Return'].std()) * np.sqrt(252)

print("\n~ RESULTADO FINAL ~")
print(f"Retorno Acumulado IA: {total_ret_algo:.2%}")
print(f"Retorno Acumulado Mercado (Benchmark): {total_ret_market:.2%}")
print(f"Sharpe Ratio IA: {sharpe_algo:.2f}")

# Gráfico
plt.figure(figsize=(12, 6))
plt.plot(df_wf['Cumulative_Return'], label='Agente IA (Walk-Forward)', color='blue')
plt.plot(market_return, label='Média Mercado', color='gray', linestyle='--')
plt.title(' TesteWalk-Forward')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#PERMUTAÇÃO -------------------------------------------------------------
# Função Wrapper para o Paralelismo
def processar_permutacao(i, features, returns_original):
    torch.set_num_threads(1)
    
    returns_shuffled = returns_original.sample(frac=1, random_state=i).reset_index(drop=True)
    returns_shuffled.index = returns_original.index 
    
    # Roda o Walk-Forward 
    score = executar_walk_forward(features, returns_shuffled, verbose=False)
    print(f"Permutação {i} finalizada.")
    return score

# Executa em Paralelo
sharpes_falsos = Parallel(n_jobs=-1)(
    delayed(processar_permutacao)(i, df_features, df_returns) 
    for i in range(20)
)

# Análise e Gráfico 
p_valor = np.mean(np.array(sharpes_falsos) >= sharpe_real)

plt.figure(figsize=(10, 6))
plt.hist(sharpes_falsos, bins=10, color='gray', alpha=0.7, label='Sorte (Dados Aleatórios)')
plt.axvline(sharpe_real, color='red', linestyle='--', linewidth=2, label='Seu Robô (Real)')
plt.title(f"Teste de Permutação Walk-Forward (P-valor: {p_valor:.3f})")
plt.xlabel("Sharpe Ratio")
plt.legend()
plt.show()

print(f"{p_valor}")